# 🇹🇭 Thai Stock Financials — ตั้งค่าและ Seed ข้อมูลครั้งแรก

โน้ตบุ๊คนี้ใช้ทำ 2 อย่าง (ทำครั้งเดียวตอนติดตั้ง หรือรันซ้ำเมื่อไหร่ก็ได้ที่ต้องการ):

1. **Seed ข้อมูล**: ดึงข้อมูลงบการเงินหุ้นชุดแรก (~100 ตัว) เข้า Supabase ทันที โดยไม่ต้องรอ GitHub Actions รอบ 24 ชม.
2. **Deploy Edge Function**: ติดตั้งฟังก์ชัน `trigger-update` ขึ้น Supabase (ทำครั้งเดียวตอนตั้งระบบ)

ทำตามลำดับเซลล์ทีละอันได้เลย ไม่ต้องมีความรู้เขียนโปรแกรม

## ขั้นที่ 1: ดึงโค้ดโปรเจกต์จาก GitHub ของคุณ

In [ ]:
GITHUB_REPO_URL = "https://github.com/YOUR-USERNAME/YOUR-REPO-NAME.git"  # แก้เป็นลิงก์ repo ของคุณ

!git clone $GITHUB_REPO_URL project
%cd project
!pip install -q -r requirements.txt

## ขั้นที่ 2: ใส่ค่า Supabase ของคุณ (หาได้จาก Project Settings > API)

In [ ]:
import os
from getpass import getpass

os.environ["SUPABASE_URL"] = input("วาง Supabase Project URL (เช่น https://xxxx.supabase.co): ").strip()
os.environ["SUPABASE_SERVICE_KEY"] = getpass("วาง Supabase service_role key (จะไม่แสดงตอนพิมพ์): ").strip()
print("บันทึกค่าเรียบร้อย")

## ขั้นที่ 3: Seed ข้อมูลชุดแรก (~100 หุ้น)
ใช้เวลาประมาณ 5-10 นาที ระหว่างรอสามารถปล่อยแท็บนี้ไว้เฉยๆ ได้เลย

In [ ]:
!python fetch_financials.py --all --seed-file tickers_seed.txt

## ขั้นที่ 4 (ทำครั้งเดียวตอนติดตั้งระบบ): Deploy Edge Function ขึ้น Supabase
ขั้นตอนนี้จะติดตั้ง Supabase CLI ชั่วคราวใน Colab (ไม่ได้ติดตั้งลงเครื่องคุณ) แล้วอัปโหลดฟังก์ชัน `trigger-update`

ก่อนรัน ให้เตรียม 3 อย่าง:
- **Supabase Access Token**: สร้างได้ที่ https://supabase.com/dashboard/account/tokens
- **Supabase Project Ref**: ดูได้จาก URL โปรเจกต์ เช่น `xxxxx` ใน `https://xxxxx.supabase.co`
- **GitHub Personal Access Token** (สิทธิ์ `repo` และ `workflow`): สร้างได้ที่ https://github.com/settings/tokens

In [ ]:
!npm install -g supabase

SUPABASE_ACCESS_TOKEN = getpass("วาง Supabase Access Token: ").strip()
PROJECT_REF = input("วาง Supabase Project Ref: ").strip()
GITHUB_TOKEN = getpass("วาง GitHub Personal Access Token: ").strip()
GITHUB_REPO = input("พิมพ์ชื่อ repo แบบเต็ม เช่น your-username/your-repo-name: ").strip()

import os
os.environ["SUPABASE_ACCESS_TOKEN"] = SUPABASE_ACCESS_TOKEN

!supabase functions deploy trigger-update --project-ref $PROJECT_REF --no-verify-jwt

# หมายเหตุ: ตั้งเฉพาะ GITHUB_TOKEN/GITHUB_REPO เท่านั้น
# SUPABASE_URL และ SUPABASE_SERVICE_ROLE_KEY เป็นชื่อสงวน Supabase ใส่ให้ Edge Function อัตโนมัติอยู่แล้ว ตั้งเองไม่ได้
!supabase secrets set GITHUB_TOKEN=$GITHUB_TOKEN GITHUB_REPO=$GITHUB_REPO --project-ref $PROJECT_REF

print("\n\u2705 Deploy เสร็จแล้ว! URL ของฟังก์ชันคือ:")
print(f"https://{PROJECT_REF}.supabase.co/functions/v1/trigger-update")
print("\nนำ URL นี้ไปวางในไฟล์ docs/config.js ช่อง TRIGGER_FUNCTION_URL")

## (ทางเลือก) อัปเดตหุ้นตัวเดียวด้วยตัวเองตอนนี้เลย
ใช้เวลาไม่กี่วินาที เหมาะเวลาอยากทดสอบหรือเพิ่มหุ้นตัวใหม่ทันที

In [ ]:
one_ticker = input("พิมพ์ ticker ที่ต้องการอัปเดต เช่น PTT: ").strip()
!python fetch_financials.py --ticker "$one_ticker"